# Clean Script

In [61]:
import pandas as pd
from datetime import datetime

# Load the CSV data
def load_data(file_path):
    df = pd.read_csv(file_path)
    df['ladate'] = pd.to_datetime(df['ladate'], format='%d/%m/%Y')
    # Extract year and month for grouping
    df['year'] = df['ladate'].dt.year
    df['month'] = df['ladate'].dt.month
    return df

# Filter data based on specified columns
def filter_data_based_on_columns(df, e_offer_values, c_data_type_values, country_list):
    return df[
        df['E_OFFER'].isin(e_offer_values) &
        df['C_DATA_TYPE'].isin(c_data_type_values) &
        df['country'].isin(country_list)
    ]

# Group and aggregate the data by key columns
def group_data(df):
    return df.groupby(['year', 'month', 'country', 'C_DATA_TYPE', 'D_BRAND', 'E_OFFER'], as_index=False).agg({
        'Digital_sales': 'sum',
        'E2E_Digital_sales': 'sum',
        'Assisted_Digital_sales': 'sum'
    })

# Calculate YoY difference
def calculate_yoy_diff(grouped_df):
    grouped_df['Digital_sales_YoY'] = grouped_df['Digital_sales'].diff(12)
    grouped_df['E2E_Digital_sales_YoY'] = grouped_df['E2E_Digital_sales'].diff(12)
    grouped_df['Assisted_Digital_sales_YoY'] = grouped_df['Assisted_Digital_sales'].diff(12)
    return grouped_df.dropna(subset=['Digital_sales_YoY', 'E2E_Digital_sales_YoY', 'Assisted_Digital_sales_YoY'])

# Find maximum YoY growth for each metric
def find_max_growth(yoy_df):
    max_growth_digital = yoy_df.loc[yoy_df.groupby(['country', 'year', 'month'])['Digital_sales_YoY'].idxmax()]
    max_growth_e2e = yoy_df.loc[yoy_df.groupby(['country', 'year', 'month'])['E2E_Digital_sales_YoY'].idxmax()]
    max_growth_assisted = yoy_df.loc[yoy_df.groupby(['country', 'year', 'month'])['Assisted_Digital_sales_YoY'].idxmax()]
    
    # Combine results for all metrics
    return pd.concat([max_growth_digital, max_growth_e2e, max_growth_assisted])

# Melt the dataframe to combine metric columns
def melt_max_growth(df):
    melted_df = df.melt(id_vars=['year', 'month', 'country', 'C_DATA_TYPE', 'D_BRAND', 'E_OFFER'],
                        value_vars=['Digital_sales_YoY', 'E2E_Digital_sales_YoY', 'Assisted_Digital_sales_YoY'],
                        var_name='metric',
                        value_name='dif_volume')
    
    # Find the row index of the maximum absolute YoY value for each metric
    max_idx = melted_df.groupby(['year', 'month', 'country', 'metric'])['dif_volume'].apply(lambda x: x.abs().idxmax())
    return melted_df.loc[max_idx]

# Main function to process the entire pipeline
def process_sales_data(file_path):
    # Step 1: Load data
    df = load_data(file_path)

    # Define filter values
    e_offer_values = ['Mobile Only postpaid', 'Fixed Only', 'Mobile Convergent postpaid', 'Fixed Convergent']
    c_data_type_values = ['Acquisitions', 'Renewals']
    columns_to_sum = ['E2E_Digital_sales', 'Assisted_Digital_sales', 'All_channels_sales']
    countries = ['osp','opl', 'obe', 'oro', 'osk', 'omd', 'olu']
    
    # Filter data
    filtered_df = filter_data_based_on_columns(df, e_offer_values, c_data_type_values, countries)
    
    # Step 2: Group data
    grouped_df = group_data(filtered_df)
    
    # Step 3: Calculate YoY difference
    yoy_results = calculate_yoy_diff(grouped_df)
    
    # Step 4: Find max growth across metrics
    max_growth_df = find_max_growth(yoy_results)
    
    # Step 5: Melt the results for better visualization
    final_result = melt_max_growth(max_growth_df)
    
    return final_result

# File path to the data
file_path = 'Raw data EUR Scorecard - Full extract.csv'

# Process the data
final_output = process_sales_data(file_path)



In [55]:
final_output.head()

,year,month,country,C_DATA_TYPE,D_BRAND,E_OFFER,metric,dif_volume
2538,2017,8,oxx,Standalone devices,Bbrand,-,Assisted_Digital_sales_YoY,0.0
354,2017,8,oxx,Standalone devices,Bbrand,-,Digital_sales_YoY,1052.0
1446,2017,8,oxx,Standalone devices,Bbrand,-,E2E_Digital_sales_YoY,233.0
2539,2017,9,oxx,Offer sales with a device,All,All,Assisted_Digital_sales_YoY,0.0
355,2017,9,oxx,Offer sales with a device,All,All,Digital_sales_YoY,-157.0


In [63]:
final_output[(final_output['year'] == 2024) & 
                             (final_output['month'] == 5) & 
                             (final_output['country'] == 'opl')]

,year,month,country,C_DATA_TYPE,D_BRAND,E_OFFER,metric,dif_volume
1458,2024,5,opl,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,3428.0
126,2024,5,opl,Renewals,Orange,Mobile Only postpaid,Digital_sales_YoY,13595.0
1014,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,E2E_Digital_sales_YoY,13119.0


# With Total

In [103]:
import pandas as pd
from datetime import datetime

# Load the CSV data
def load_data(file_path):
    df = pd.read_csv(file_path)
    df['ladate'] = pd.to_datetime(df['ladate'], format='%d/%m/%Y')
    # Extract year and month for grouping
    df['year'] = df['ladate'].dt.year
    df['month'] = df['ladate'].dt.month
    return df

# Filter data based on specified columns
def filter_data_based_on_columns(df, e_offer_values, c_data_type_values, country_list):
    return df[
        df['E_OFFER'].isin(e_offer_values) &
        df['C_DATA_TYPE'].isin(c_data_type_values) &
        df['country'].isin(country_list)
    ]

# Group and aggregate the data by key columns
def group_data(df):
    return df.groupby(['year', 'month', 'country', 'C_DATA_TYPE', 'D_BRAND', 'E_OFFER'], as_index=False).agg({
        'Digital_sales': 'sum',
        'E2E_Digital_sales': 'sum',
        'Assisted_Digital_sales': 'sum'
    })

# Calculate YoY difference
def calculate_yoy_diff(grouped_df):
    grouped_df['Digital_sales_YoY'] = grouped_df['Digital_sales'].diff(12)
    grouped_df['E2E_Digital_sales_YoY'] = grouped_df['E2E_Digital_sales'].diff(12)
    grouped_df['Assisted_Digital_sales_YoY'] = grouped_df['Assisted_Digital_sales'].diff(12)
    
    return grouped_df.dropna(subset=['Digital_sales_YoY', 'E2E_Digital_sales_YoY', 'Assisted_Digital_sales_YoY'])

# Find maximum YoY growth for each metric
def find_max_growth(yoy_df):
    max_growth_digital = yoy_df.loc[yoy_df.groupby(['country', 'year', 'month'])['Digital_sales_YoY'].idxmax()]
    max_growth_e2e = yoy_df.loc[yoy_df.groupby(['country', 'year', 'month'])['E2E_Digital_sales_YoY'].idxmax()]
    max_growth_assisted = yoy_df.loc[yoy_df.groupby(['country', 'year', 'month'])['Assisted_Digital_sales_YoY'].idxmax()]
    
    # Combine results for all metrics
    return pd.concat([max_growth_digital, max_growth_e2e, max_growth_assisted]).drop_duplicates()

# Melt the dataframe to combine metric columns
def melt_max_growth(df):
    melted_df = df.melt(id_vars=['year', 'month', 'country', 'C_DATA_TYPE', 'D_BRAND', 'E_OFFER'],
                        value_vars=['Digital_sales_YoY', 'E2E_Digital_sales_YoY', 'Assisted_Digital_sales_YoY'],
                        var_name='metric',
                        value_name='growth_value')

    # Check if melted_df is not empty before grouping
    if melted_df.empty:
        return pd.DataFrame(columns=melted_df.columns)

    # Find the row index of the maximum YoY value for each metric
    max_idx = melted_df.groupby(['year', 'month', 'country', 'metric'])['growth_value'].idxmax()
    
    if max_idx.empty:
        return pd.DataFrame(columns=melted_df.columns)
    
    return melted_df.loc[max_idx]

# Add a row for Europe with the max growth values
def add_europe_row(final_df):
    # Create a list to hold Europe data
    europe_data_list = []

    # Iterate through unique years, months, and metrics to find max growth for Europe
    for (year, month, metric) in final_df.groupby(['year', 'month', 'metric']).groups.keys():
        # Filter the data for each combination of year, month, and metric
        subset = final_df[(final_df['year'] == year) & (final_df['month'] == month) & (final_df['metric'] == metric)]
        
        if not subset.empty:
            # Find the row with the maximum growth value
            max_row = subset.loc[subset['growth_value'].idxmax()]  # Use 'dif_volume' instead of 'growth_value'
            europe_data_list.append(max_row)

    # Create a DataFrame from the list
    europe_data = pd.DataFrame(europe_data_list)

    # Add the country column as Europe
    europe_data['country'] = 'Europe'
    
    # Reorder the columns to match the final DataFrame
    europe_data = europe_data[final_df.columns]

    # Concatenate the Europe data with the final DataFrame
    return pd.concat([final_df, europe_data], ignore_index=True)
# Main function to process the entire pipeline
def process_sales_data(file_path):
    # Step 1: Load data
    df = load_data(file_path)

    # Define filter values
    e_offer_values = ['Mobile Only postpaid', 'Fixed Only', 'Mobile Convergent postpaid', 'Fixed Convergent']
    c_data_type_values = ['Acquisitions', 'Renewals']
    countries = ['osp','opl', 'obe', 'oro', 'osk', 'omd', 'olu']
    
    # Filter data
    filtered_df = filter_data_based_on_columns(df, e_offer_values, c_data_type_values, countries)
    
    # Step 2: Group data
    grouped_df = group_data(filtered_df)
    
    # Step 3: Calculate YoY difference
    yoy_results = calculate_yoy_diff(grouped_df)
    
    # Step 4: Find max growth across metrics
    max_growth_df = find_max_growth(yoy_results)
    
    # Step 5: Melt the results for better visualization
    final_result = melt_max_growth(max_growth_df)
    
    # Step 6: Add Europe row
    final_result_with_europe = add_europe_row(final_result)
    
    return final_result_with_europe

# File path to the data
file_path = 'Raw data EUR Scorecard - Full extract.csv'

# Process the data
final_output = process_sales_data(file_path)



In [104]:
final_output[(final_output['year'] == 2024) & 
                             (final_output['month'] == 5)  
                            ]

,year,month,country,C_DATA_TYPE,D_BRAND,E_OFFER,metric,growth_value
648,2024,5,obe,Acquisitions,Orange,Mobile Convergent postpaid,Assisted_Digital_sales_YoY,1411.0
649,2024,5,obe,Acquisitions,Orange,Mobile Only postpaid,Digital_sales_YoY,2061.0
650,2024,5,obe,Acquisitions,Orange,Mobile Only postpaid,E2E_Digital_sales_YoY,2018.0
651,2024,5,olu,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,1.0
652,2024,5,olu,Acquisitions,Orange,Fixed Only,Digital_sales_YoY,-107.0
653,2024,5,olu,Acquisitions,Orange,Fixed Only,E2E_Digital_sales_YoY,-19.0
654,2024,5,omd,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,617.0
655,2024,5,omd,Acquisitions,Orange,Fixed Only,Digital_sales_YoY,-775.0
656,2024,5,omd,Acquisitions,Orange,Fixed Only,E2E_Digital_sales_YoY,0.0
657,2024,5,opl,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,3428.0


In [107]:
# final_output[(final_output['year'] == 2024) & (final_output['month'] == 5) & (final_output['metric'] == 'E2E_Digital_sales_YoY')].sort_values(by=['growth_value'], ascending=False)

,year,month,country,C_DATA_TYPE,D_BRAND,E_OFFER,metric,growth_value
659,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,E2E_Digital_sales_YoY,13119.0
785,2024,5,Europe,Acquisitions,Flex by Orange,Mobile Only postpaid,E2E_Digital_sales_YoY,13119.0
662,2024,5,oro,Acquisitions,Yoxo,Mobile Only postpaid,E2E_Digital_sales_YoY,10845.0
650,2024,5,obe,Acquisitions,Orange,Mobile Only postpaid,E2E_Digital_sales_YoY,2018.0
665,2024,5,osk,Acquisitions,Orange,Mobile Only postpaid,E2E_Digital_sales_YoY,273.0
656,2024,5,omd,Acquisitions,Orange,Fixed Only,E2E_Digital_sales_YoY,0.0
653,2024,5,olu,Acquisitions,Orange,Fixed Only,E2E_Digital_sales_YoY,-19.0


In [120]:

def filter_data_for_month_year(df, month, year):
    return df[(df['ladate'].dt.year == year) & (df['ladate'].dt.month == month)]

def filter_data_based_on_columns(df, e_offer_values, c_data_type_values, country_list):
    return df[
        df['E_OFFER'].isin(e_offer_values) &
        df['C_DATA_TYPE'].isin(c_data_type_values) &
        df['country'].isin(country_list)
    ]

def calculate_sums_and_shares(df, columns_to_sum, month, year):
    sum_by_country = df.groupby('country')[columns_to_sum].sum().reset_index()
    sum_by_country['E2E_Digital_share'] = ((sum_by_country['E2E_Digital_sales'] / sum_by_country['All_channels_sales']) * 100).round(1)
    sum_by_country['Assisted_Digital_share'] = ((sum_by_country['Assisted_Digital_sales'] / sum_by_country['All_channels_sales']) * 100).round(1)
    sum_by_country['share_digital_all'] = (((sum_by_country['E2E_Digital_sales'] + sum_by_country['Assisted_Digital_sales']) / sum_by_country['All_channels_sales']) * 100).round(1)

    # Assign month and year to the dataframe
    sum_by_country['month'] = month
    sum_by_country['year'] = year
    return sum_by_country

def add_total_row(sum_by_country, columns_to_sum, month, year):
    # Sum the specified columns across all countries
    total_sum = sum_by_country[columns_to_sum].sum()
    # Calculate the total E2E Digital share percentage
    total_e2e_digital_share = (total_sum['E2E_Digital_sales'] / total_sum['All_channels_sales']) * 100
    # Calculate the total Assisted Digital share percentage
    total_assisted_digital_share = (total_sum['Assisted_Digital_sales'] / total_sum['All_channels_sales']) * 100
    # Calculate the total Digital share percentage
    total_share_digital_all = ((total_sum['E2E_Digital_sales'] + total_sum['Assisted_Digital_sales']) / total_sum['All_channels_sales']) * 100

    # Create a dataframe for the total row
    total_row = pd.DataFrame(data={
        'E2E_Digital_sales': [total_sum['E2E_Digital_sales']],
        'Assisted_Digital_sales': [total_sum['Assisted_Digital_sales']],
        'All_channels_sales': [total_sum['All_channels_sales']],
        'E2E_Digital_share': [round(total_e2e_digital_share, 1)],
        'Assisted_Digital_share': [round(total_assisted_digital_share, 1)],
        'share_digital_all': [round(total_share_digital_all, 1)],
        'month': [month],
        'year': [year],
        'country': 'Europe'
    })

    # Concatenate the total row with the original dataframe
    sum_by_country_with_total = pd.concat([sum_by_country, total_row])

    return sum_by_country_with_total

def process_data(df, e_offer_values, c_data_type_values, columns_to_sum, country_list):
    results = []
    df = load_data(file_path)
    # Get unique months and years from the dataframe
    unique_months_years = df[['ladate']].apply(lambda x: (x['ladate'].month, x['ladate'].year), axis=1).unique()

    for month, year in unique_months_years:
        # Filter data for the current month and year
        filtered_df = filter_data_for_month_year(df, month, year)
        filtered_df = filter_data_based_on_columns(filtered_df, e_offer_values, c_data_type_values, country_list)
        sum_by_country = calculate_sums_and_shares(filtered_df, columns_to_sum, month, year)
        sum_by_country_with_total = add_total_row(sum_by_country, columns_to_sum, month, year)
        results.append(sum_by_country_with_total)
        
    # Concatenate all results into a single dataframe
    final_shares = pd.concat(results)

    return final_shares


In [139]:

shares = process_data(df, e_offer_values, c_data_type_values, columns_to_sum, countries)


C:\Users\YZLT0221\AppData\Local\Temp\1\ipykernel_16692\1558072045.py:26: RuntimeWarning: invalid value encountered in scalar divide
  total_e2e_digital_share = (total_sum['E2E_Digital_sales'] / total_sum['All_channels_sales']) * 100
C:\Users\YZLT0221\AppData\Local\Temp\1\ipykernel_16692\1558072045.py:28: RuntimeWarning: invalid value encountered in scalar divide
  total_assisted_digital_share = (total_sum['Assisted_Digital_sales'] / total_sum['All_channels_sales']) * 100
C:\Users\YZLT0221\AppData\Local\Temp\1\ipykernel_16692\1558072045.py:30: RuntimeWarning: invalid value encountered in scalar divide
  total_share_digital_all = ((total_sum['E2E_Digital_sales'] + total_sum['Assisted_Digital_sales']) / total_sum['All_channels_sales']) * 100
C:\Users\YZLT0221\AppData\Local\Temp\1\ipykernel_16692\1558072045.py:26: RuntimeWarning: invalid value encountered in scalar divide
  total_e2e_digital_share = (total_sum['E2E_Digital_sales'] / total_sum['All_channels_sales']) * 100
C:\Users\YZLT0221\

In [140]:
 shares[(shares['month'] == 5) & (shares['year'] == 2024)]

,country,E2E_Digital_sales,Assisted_Digital_sales,All_channels_sales,E2E_Digital_share,Assisted_Digital_share,share_digital_all,month,year
0,obe,7860.0,4763.0,58575.0,13.4,8.1,21.6,5,2024
1,olu,197.0,32.0,2449.0,8.0,1.3,9.4,5,2024
2,omd,0.0,1091.0,19012.0,0.0,5.7,5.7,5,2024
3,opl,47471.0,13541.0,225720.0,21.0,6.0,27.0,5,2024
4,oro,21623.0,1034.0,172395.0,12.5,0.6,13.1,5,2024
5,osk,2337.0,1766.0,37546.0,6.2,4.7,10.9,5,2024
0,Europe,79488.0,22227.0,515697.0,15.4,4.3,19.7,5,2024


In [123]:
merged_output = pd.merge(final_output, shares, on=['year', 'month', 'country'], how='left')


In [125]:
 merged_output[(merged_output['month'] == 5) & (merged_output['year'] == 2024)& (merged_output['metric'] == 'Assisted_Digital_sales_YoY')]

,year,month,country,C_DATA_TYPE,D_BRAND,E_OFFER,metric,growth_value,E2E_Digital_sales,Assisted_Digital_sales,All_channels_sales,E2E_Digital_share,Assisted_Digital_share,share_digital_all
648,2024,5,obe,Acquisitions,Orange,Mobile Convergent postpaid,Assisted_Digital_sales_YoY,1411.0,7860.0,4763.0,58575.0,13.4,8.1,21.6
651,2024,5,olu,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,1.0,197.0,32.0,2449.0,8.0,1.3,9.4
654,2024,5,omd,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,617.0,0.0,1091.0,19012.0,0.0,5.7,5.7
657,2024,5,opl,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,3428.0,47471.0,13541.0,225720.0,21.0,6.0,27.0
660,2024,5,oro,Acquisitions,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,526.0,21623.0,1034.0,172395.0,12.5,0.6,13.1
663,2024,5,osk,Acquisitions,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,-51.0,2337.0,1766.0,37546.0,6.2,4.7,10.9
783,2024,5,Europe,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,3428.0,79488.0,22227.0,515697.0,15.4,4.3,19.7


In [132]:
# Calculate growth_percentage
merged_output['growth_percentage'] = merged_output['growth_value'] / merged_output['All_channels_sales']*100

# To handle division by zero, you might want to replace any infinite values or NaNs
merged_output['growth_percentage'] = merged_output['growth_percentage'].replace([float('inf'), -float('inf')], float('nan'))

growth_value_index = merged_output.columns.get_loc('growth_value')  # Get the index of the growth_value column
merged_output.insert(growth_value_index + 1, 'growth_percentage', merged_output.pop('growth_percentage'))


In [133]:
merged_output

,year,month,country,C_DATA_TYPE,D_BRAND,E_OFFER,metric,growth_value,growth_percentage,E2E_Digital_sales,Assisted_Digital_sales,All_channels_sales,E2E_Digital_share,Assisted_Digital_share,share_digital_all
0,2021,2,obe,Acquisitions,Orange,Mobile Convergent postpaid,Assisted_Digital_sales_YoY,1530.0,2.860348,5117.0,4465.0,53490.0,9.6,8.3,17.9
1,2021,2,obe,Acquisitions,Orange,Fixed Only,Digital_sales_YoY,739.0,1.381567,5117.0,4465.0,53490.0,9.6,8.3,17.9
2,2021,2,obe,Acquisitions,Orange,Mobile Only postpaid,E2E_Digital_sales_YoY,-352.0,-0.658067,5117.0,4465.0,53490.0,9.6,8.3,17.9
3,2021,2,oro,Acquisitions,Yoxo,Mobile Only postpaid,Assisted_Digital_sales_YoY,0.0,0.000000,3294.0,0.0,3294.0,100.0,0.0,100.0
4,2021,2,oro,Acquisitions,Yoxo,Mobile Only postpaid,Digital_sales_YoY,-1458.0,-44.262295,3294.0,0.0,3294.0,100.0,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
781,2024,4,Europe,Renewals,Orange,Mobile Only postpaid,Digital_sales_YoY,13802.0,2.512762,81814.0,24727.0,549276.0,14.9,4.5,19.4
782,2024,4,Europe,Acquisitions,Flex by Orange,Mobile Only postpaid,E2E_Digital_sales_YoY,13516.0,2.460694,81814.0,24727.0,549276.0,14.9,4.5,19.4
783,2024,5,Europe,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY,3428.0,0.664731,79488.0,22227.0,515697.0,15.4,4.3,19.7
784,2024,5,Europe,Renewals,Orange,Mobile Only postpaid,Digital_sales_YoY,13595.0,2.636238,79488.0,22227.0,515697.0,15.4,4.3,19.7


In [138]:
shares.head()

,country,E2E_Digital_sales,Assisted_Digital_sales,All_channels_sales,E2E_Digital_share,Assisted_Digital_share,share_digital_all,month,year,date
0,obe,6695.0,5632.0,62992.0,10.6,8.9,19.6,8,2022,2022-08-01
1,olu,367.0,0.0,2479.0,14.8,0.0,14.8,8,2022,2022-08-01
2,omd,0.0,509.0,24256.0,0.0,2.1,2.1,8,2022,2022-08-01
3,opl,38695.0,11070.0,258050.0,15.0,4.3,19.3,8,2022,2022-08-01
4,oro,10343.0,1259.0,161987.0,6.4,0.8,7.2,8,2022,2022-08-01


In [154]:
df1 = shares.sort_values(by='date', ascending=False)

# Generate comments
comments = []
share_columns = ['E2E_Digital_share', 'Assisted_Digital_share', 'share_digital_all']

for idx, row in df1.iterrows():
    for share_col in share_columns:
        current_value = row[share_col]
        previous_year = row['year'] - 1
        
        # Select the previous year's row with the same month
        previous_value_row = df1[(df1['month'] == row['month']) & (df1['year'] == previous_year) & (df1['country'] == row['country'])]
        
        country = row['country'].upper()
        if not previous_value_row.empty:
            previous_value = previous_value_row[share_col].iloc[0]  # Use .iloc to safely select the value
            difference = current_value - previous_value  # Calculate the difference
            comments.append((row['month'], row['year'], country, share_col, current_value, previous_value))

In [155]:
comments[:6]

[(5, 2024, 'OBE', 'E2E_Digital_share', 13.4, 11.1),
 (5, 2024, 'OBE', 'Assisted_Digital_share', 8.1, 7.3),
 (5, 2024, 'OBE', 'share_digital_all', 21.6, 18.3),
 (5, 2024, 'EUROPE', 'E2E_Digital_share', 15.4, 12.1),
 (5, 2024, 'EUROPE', 'Assisted_Digital_share', 4.3, 3.4),
 (5, 2024, 'EUROPE', 'share_digital_all', 19.7, 15.4)]

# Comments

In [ ]:
import pandas as pd
from google.cloud import bigquery
import openai

# Set up the Azure OpenAI configuration
openai.api_type = "azure"
openai.api_base = "https://open-ai-eur-europe-sbx-weu.openai.azure.com/"
openai.api_key = "e8f065cf102a4988b3b4f6ad60864005"
openai.api_version = "2023-09-15-preview"

def generate_comment_for_share(month, year, country, column_name, difference, growth_percentage, c_data_type, d_brand, e_offer):
    # Determine the strength and type of change based on the difference
    if difference > 0.5:
        change_strength = "strong increase"
    elif difference < -0.5:
        change_strength = "strong decrease"
    elif 0 < difference <= 0.5:
        change_strength = "slight increase"
    elif -0.5 < difference < 0:
        change_strength = "slight decrease"
    else:
        change_strength = "stable"

    # Adjust the country name for "Total"
    country_name = country if country != "Europe" else "most countries"

    prompt_ = (f"There is a {change_strength} of {abs(growth_percentage):.2f}% YoY for {column_name.replace('_', ' ')} "
                f"in {country_name} for the month of {month} {year}, where {c_data_type}/{d_brand}/{e_offer} is the most contributor with growth of {growth_percentage:.2f}%.")

    messages = [
        {"role": "system", "content": "You will generate a single sentence comment based on the given data, using the terms 'strong/slight increase/decrease' or 'stable'."},
        {"role": "user", "content": prompt_}
    ]
    
    # Send a completion call to Azure OpenAI to generate a comment
    response = openai.ChatCompletion.create(
        engine="eur-europe-sbx-open-ai-automated-deployment-weu",
        messages=messages,
        max_tokens=30,
        temperature=0.2
    )

    return response['choices'][0]['message']['content']


def main(request):
    # Define BigQuery table to retrieve the processed data
    project_id = 'eur-itnanalytics-97661-sbx'
    dataset_id = 'EUR_Digital_Scorecard'
    table_id = 'Digital_E2E_YoY_Data'  
    table_full = f'{project_id}.{dataset_id}.{table_id}'
    dist_table_id = 'Digital E2E -YoY comments'
    dist_full = f'{project_id}.{dataset_id}.{dist_table_id}'

    # Initialize BigQuery client
    client = bigquery.Client()

    # Construct a BigQuery SQL query to fetch your data
    query = f"""
        SELECT *
        FROM `{table_full}`;
    """
    df = client.query(query).to_dataframe()
    print('Start loading the data...')

    # Convert 'month' and 'year' to datetime
    df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))

    # Sort the dataframe by date
    df = df.sort_values(by='date', ascending=[False])

    # Define a mapping between share_column and metric
    share_column_to_metric = {
    'Assisted_Digital_share': 'Assisted_Digital_sales_YoY',
    'E2E_Digital_share': 'E2E_Digital_sales_YoY',
    'share_digital_all': 'Digital_sales_YoY'}

    # Generate comments
    comments = []
    share_columns = ['E2E_Digital_share', 'Assisted_Digital_share', 'share_digital_all']

    for idx, row in df.iterrows():
        for share_col in share_columns:
            current_value = row[share_col]
            previous_year = row['year'] - 1
            previous_value_row = df[(df['month'] == row['month']) & (df['year'] == previous_year) & (df['country'] == row['country'])]
            country = row['country'].upper()
            
            if not previous_value_row.empty:
                previous_value = previous_value_row[share_col].iloc[0]
                difference = current_value - previous_value  # Calculate the difference
                
                # Retrieve the corresponding metric based on the share_column
                metric = share_column_to_metric[share_col]
                
                # Use growth_percentage for comment generation
                growth_percentage = row['growth_percentage']
                comment = generate_comment_for_share(row['month'], row['year'], country, share_col, difference, growth_percentage, row['C_DATA_TYPE'], row['D_BRAND'], row['E_OFFER'])
                
                # Append new columns including the metric
                comments.append((row['month'], row['year'], country, share_col, current_value, previous_value, comment,
                                row['C_DATA_TYPE'], row['D_BRAND'], row['E_OFFER'], growth_percentage, metric))
        
        print("Sub step!")
    
    print(comments[:5])
    # Create a DataFrame for the comments
    comments_df = pd.DataFrame(comments, columns=['month', 'year', 'country', 'share_column', 'current_month_value', 
                                              'YoY_month_value', 'comment', 'C_DATA_TYPE', 'D_BRAND', 'E_OFFER', 
                                              'growth_percentage', 'metric'])

    job_config = bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField("month", "INTEGER", mode="REQUIRED"),
            bigquery.SchemaField("year", "INTEGER", mode="REQUIRED"),
            bigquery.SchemaField("country", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("share_column", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("current_month_value", "FLOAT", mode="NULLABLE"),
            bigquery.SchemaField("YoY_month_value", "FLOAT", mode="NULLABLE"),
            bigquery.SchemaField("comment", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("C_DATA_TYPE", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("D_BRAND", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("E_OFFER", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("growth_percentage", "FLOAT", mode="NULLABLE"),
            bigquery.SchemaField("metric", "STRING", mode="NULLABLE")
        ],
        write_disposition='WRITE_TRUNCATE',  # Other options: WRITE_APPEND, WRITE_EMPTY
    )

    # Save the processed data back to BigQuery
    job = client.load_table_from_dataframe(comments_df, dist_full, job_config=job_config)
    job.result()  # Wait for the job to complete

    table = client.get_table(dist_full)  # Make an API request
    print(f"Loaded {table.num_rows} rows and {len(table.schema)} columns to {dist_full}")

    return "Done!"


# Without functions to test

In [50]:
import pandas as pd
from datetime import datetime

In [51]:
# Load the CSV file
file_path = 'Raw data EUR Scorecard - Full extract.csv'  # Replace with the actual file path
df = pd.read_csv(file_path)

In [52]:
df.head()

,ladate,country,C_DATA_TYPE,D_BRAND,E_OFFER,Digital_sales,E2E_Digital_sales,Assisted_Digital_sales,Pick_up_in_store,All_channels_sales,Digital_share,E2E_Digital_share,Assisted_Digital_share
0,01/08/2022,osp,TV contracts acquisitions,Orange,TV,8232.0,2436.0,5796.0,NaN,29707.0,0.277106,0.082001,0.195106
1,01/07/2020,osp,TV contracts acquisitions,Orange,TV,2739.0,296.0,2443.0,NaN,12763.0,0.214605,0.023192,0.191413
2,01/11/2023,osp,TV contracts acquisitions,Orange,TV,3754.0,1107.0,2647.0,NaN,15407.0,0.243655,0.071850,0.171805
3,01/04/2023,osp,TV contracts acquisitions,Orange,TV,3077.0,985.0,2092.0,NaN,14855.0,0.207136,0.066308,0.140828
4,01/12/2020,osp,TV contracts acquisitions,Orange,TV,3529.0,378.0,3151.0,NaN,14759.0,0.239108,0.025611,0.213497


In [53]:
df['country'].unique()

array(['osp', 'opl', 'olu', 'osk', 'omd', 'oxx', 'obe', 'ofr', 'oro'],
      dtype=object)

In [4]:
df['ladate'] = pd.to_datetime(df['ladate'], format='%d/%m/%Y')


In [5]:
# Extract year and month for grouping
df['year'] = df['ladate'].dt.year
df['month'] = df['ladate'].dt.month

# Group by year, month, country, C_DATA_TYPE, D_BRAND, E_OFFER
grouped = df.groupby(['year', 'month', 'country', 'C_DATA_TYPE', 'D_BRAND', 'E_OFFER'], as_index=False).agg({
    'Digital_sales': 'sum',
    'E2E_Digital_sales': 'sum',
    'Assisted_Digital_sales': 'sum'
})

# Calculate YoY difference (adjusted without using previous year columns)
grouped['Digital_sales_YoY_diff'] = grouped['Digital_sales'].diff(12)
grouped['E2E_Digital_sales_YoY_diff'] = grouped['E2E_Digital_sales'].diff(12)
grouped['Assisted_Digital_sales_YoY_diff'] = grouped['Assisted_Digital_sales'].diff(12)

# Filter for only rows with valid YoY comparisons
yoy_results = grouped.dropna(subset=['Digital_sales_YoY_diff', 'E2E_Digital_sales_YoY_diff', 'Assisted_Digital_sales_YoY_diff'])

# Find the highest YoY growth for each metric
max_growth_digital = yoy_results.loc[yoy_results.groupby(['country', 'year', 'month'])['Digital_sales_YoY_diff'].idxmax()]
max_growth_e2e = yoy_results.loc[yoy_results.groupby(['country', 'year', 'month'])['E2E_Digital_sales_YoY_diff'].idxmax()]
max_growth_assisted = yoy_results.loc[yoy_results.groupby(['country', 'year', 'month'])['Assisted_Digital_sales_YoY_diff'].idxmax()]

# Combine the results into one DataFrame
max_growth_combined = pd.concat([max_growth_digital, max_growth_e2e, max_growth_assisted])

# Select relevant columns for the final output
result_df = max_growth_combined[['year', 'month', 'country', 'C_DATA_TYPE', 'D_BRAND', 'E_OFFER',
                                'Digital_sales_YoY_diff', 'E2E_Digital_sales_YoY_diff', 'Assisted_Digital_sales_YoY_diff']]


In [6]:
result_df
result_df[(result_df['year'] == 2024) & 
                             (result_df['month'] == 5) & 
                             (result_df['country'] == 'opl')]

,year,month,country,C_DATA_TYPE,D_BRAND,E_OFFER,Digital_sales_YoY_diff,E2E_Digital_sales_YoY_diff,Assisted_Digital_sales_YoY_diff
2831,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,13112.0,13119.0,0.0
2831,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,13112.0,13119.0,0.0
2843,2024,5,opl,Renewals,Orange,Mobile Only postpaid,495.0,-2933.0,3428.0


In [7]:
melted_df = result_df.melt(id_vars=['year', 'month', 'country', 'C_DATA_TYPE', 'D_BRAND', 'E_OFFER'],
                     value_vars=['Digital_sales_YoY_diff', 'E2E_Digital_sales_YoY_diff', 'Assisted_Digital_sales_YoY_diff'],
                     var_name='metric',
                     value_name='value')

# Display the reshaped DataFrame
#print(melted_df)

In [8]:
## Test May 2024
melted_df[(melted_df['year'] == 2024) & 
                             (melted_df['month'] == 5) & 
                             (melted_df['country'] == 'opl')]


,year,month,country,C_DATA_TYPE,D_BRAND,E_OFFER,metric,value
214,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,Digital_sales_YoY_diff,13112.0
578,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,Digital_sales_YoY_diff,13112.0
942,2024,5,opl,Renewals,Orange,Mobile Only postpaid,Digital_sales_YoY_diff,495.0
1306,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,E2E_Digital_sales_YoY_diff,13119.0
1670,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,E2E_Digital_sales_YoY_diff,13119.0
2034,2024,5,opl,Renewals,Orange,Mobile Only postpaid,E2E_Digital_sales_YoY_diff,-2933.0
2398,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY_diff,0.0
2762,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY_diff,0.0
3126,2024,5,opl,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY_diff,3428.0


In [12]:
max_idx = melted_df.groupby(['year', 'month', 'country', 'metric'])['value'].apply(lambda x: x.abs().idxmax())

# Step 3: Subset the melted DataFrame using the indices of the max absolute values
max_metrics_df = melted_df.loc[max_idx]

In [13]:
## Test May 2024
max_metrics_df[(max_metrics_df['year'] == 2024) & 
                             (max_metrics_df['month'] == 5) & 
                             (max_metrics_df['country'] == 'opl')]


,year,month,country,C_DATA_TYPE,D_BRAND,E_OFFER,metric,value
3126,2024,5,opl,Renewals,Orange,Mobile Only postpaid,Assisted_Digital_sales_YoY_diff,3428.0
214,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,Digital_sales_YoY_diff,13112.0
1306,2024,5,opl,Acquisitions,Flex by Orange,Mobile Only postpaid,E2E_Digital_sales_YoY_diff,13119.0
